# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all entities by their `@id` for consistency and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

For more information, see the publication: 
_Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers._

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show basic metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets and fields with their corresponding `@id`s.

In [ ]:
# List all record sets and their fields by @id
print("Available record sets:\n-------------------")
record_sets = list(dataset.record_sets())

for rs in record_sets:
    print(f"@id: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    print(f"  Description: {rs.get('description')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields (by @id):")
        for fld in fields:
            if isinstance(fld, dict):
                print(f"    {fld.get('@id')}")
            else:
                print(f"    {fld}")
    print("-------------------")

# As a preview, show first few records from the main record set
if record_sets:
    main_record_set_id = record_sets[0]["@id"]
    print(f"\nSample records for record set: {main_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print({k: v for k, v in rec.items() if not k.startswith('_')})
        if i >= 2:
            break

## 3. Data Extraction
Load each record set into a pandas DataFrame for downstream analysis. All references are by `@id`.

In [ ]:
# Build DataFrames for all record sets using their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]  # List of all record set @id

for rs_id in record_set_ids:
    print(f"Loading records from record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  DataFrame shape: {df.shape}")

# Choose main record set for analysis (if you have multiple, select appropriately)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in DataFrame for '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nFirst five rows:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Now, perform basic EDA:
- Filter records on a numeric field (by its `@id` as shown above).
- Normalize the field, and group by a related field if appropriate.

**Note:** You should choose actual field `@id`s from the output above. Here, we use an example field id typical for mass spectrometry datasets (update as needed).

In [ ]:
# Example usage - replace these variables based on prior output with real @id's

numeric_field_id = None  # e.g., '@id: cr:field/molecular_weight'
group_field_id = None    # e.g., '@id: cr:field/sample_type'
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Automatically try to select a numeric field if not already specified
    if numeric_field_id is None:
        # Try to guess a numeric field by name
        for col in df.columns:
            if any(k in col.lower() for k in ["abundance", "count", "weight", "coverage", "mw", "peptide"]):
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field_id = col
                    break
    if not numeric_field_id:
        numeric_cols = df.select_dtypes(include='number').columns.tolist()
        if numeric_cols:
            numeric_field_id = numeric_cols[0]

    print(f"Using numeric field for filtering: {numeric_field_id}")

    # Filter by threshold (example: values > 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id in df.columns else df
    print(f"Filtered records with {numeric_field_id} > {threshold} (if applicable): {filtered_df.shape[0]} rows")
    display(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    if numeric_field_id in filtered_df.columns:
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, norm_col]].head())

    # Guess a group field (e.g., protein family/sample type)
    group_field_id = None
    for col in df.columns:
        if any(k in col.lower() for k in ["description", "type", "class", "group", "condition", "sample"]):
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field is available, do a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you have loaded the FAIR² mass spectrometry dataset using its Croissant schema, explored its structure (using only `@id` references), extracted data for analysis, performed normalization and grouping operations, and visualized the results. Such workflows enable reproducible, standards-based data science with complex, FAIR metadata-linked datasets.

**Next steps:**
- Try different fields (by their `@id`) for filtering, normalization, or grouping.
- Extend the EDA to analyze protein abundance, peptide counts, or other key variables.
- For further processing, see [mlcroissant documentation](https://mlcommons.github.io/croissant/).
